# İlk günlük tahmin sonuçları
Önce `uv sync --locked --extra research --extra notebooks` ve `python scripts/run_daily_forecast.py` çalıştırılır. Bu notebook hazır koşumu okur.
Deney yayımlanmış işlenmiş seri üzerinde geriye dönüktür; saatlik erişim varsayımsaldır. Kaynak: Schaffer ve diğerleri (2025), DOI 10.5281/zenodo.15180141, CC BY 4.0.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

from IPython.display import Image, Markdown, display
from urban_water.db import PROJECT_ROOT
from urban_water_research.daily_benchmark import verify_run

latest = json.loads(
    (PROJECT_ROOT / "var/experiments/aalborg-daily-v1/latest.json").read_text(encoding="utf-8")
)
directory = Path(latest["directory"])
verify_run(directory)

In [ ]:
metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
rows = [r for r in metrics["pooled"] if r["split"] == "test" and r["issuance_hour"] == 23]
table = ["| Scope | Method | n | MAE (L) | RMSE (L) | WAPE |", "|---|---|---:|---:|---:|---:|"]
for row in rows:
    wape = f"{row['wape']:.2%}" if row["wape"] is not None else "undefined"
    table.append(
        f"| {row['scope']} | {row['method']} | {row['n']} | "
        f"{row['mae_liters']:.2f} | {row['rmse_liters']:.2f} | {wape} |"
    )
display(Markdown("\n".join(table)))

In [ ]:
display(Image(filename=str(directory / "test-mae.png")))

In [ ]:
folds = json.loads((directory / "folds.json").read_text(encoding="utf-8"))
for fold in folds:
    assert datetime.fromisoformat(
        fold["latest_training_label_available_at"]
    ) < datetime.fromisoformat(fold["origin"])
    assert not set(fold["training_keys"]) & set(fold["evaluation_keys"])
    if fold["scope"] == "household_holdout":
        assert not {"AALBORG-D-09", "AALBORG-D-10"} & set(fold["training_households"])
{
    "folds_checked": len(folds),
    "model_files": len(list((directory / "models").glob("*.joblib"))),
    "inventory": latest["inventory"],
}

Gün içi bilgi, bilinen hanelerde küçük bir ek iyileşme sağladı; ayrılan iki hanede geçmiş modelini iyileştirmedi. Aralık modeli, önceden belirlenmiş genişleyen köken düzeninde yalnız o zamana kadar tamamlanmış Kasım hedeflerini kullanabilir. Test skoruyla ayar değiştirilmedi. Saatlik tahmin ve bağlamlı varyantlar bu günlük deneyin dışında kalır.